# RAFT-UP: all 14 adjacent pairs of randomly rotated Deep-STARmap sections

**Not executed.** Run with `raftup_final_verify`. Input: 15 independently rotated sections. Each pair uses SCANIT cost and **technology="merfish"**, with no test router or ground-truth transforms. Anatomical labels enter evaluation only.

Four metrics: **LAA** (FUSEmap_main_level, strongest match per nonzero row, skip zero rows), **GPR100** (GPR_original formula, radius 100 source coordinate units, P_cut=1e-19), **coverage A**, **coverage B**. GPR radius is configurable; it is not assumed to be micrometers. Empty alignments get zero coverage and undefined LAA/GPR.

Each pair saves a dense matrix, sparse copy, cell order, cutoff decision, timings, four metrics, and an alignment PNG. The loop resumes existing matrices only when input/configuration matches. Cost caching is optional and off by default; dense matrix files and intermediate computations can require substantial disk/RAM at these sizes. Failures are recorded per pair; inspect the summary before reconstruction.

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, gc, contextlib
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.spatial import cKDTree
ROOT = Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/2_3d_reconstruction_exp/Deep-STARmap')
DATA_DIR = ROOT / "processed/synthetic_10um_random_z_rotation_seed42"
RUN_DIR = Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/deep_starmap')
FILES = sorted(DATA_DIR.glob("section_*.h5ad"))
assert len(FILES) == 15, f"Expected 15 sections, found {len(FILES)}"
PAIRS = list(zip(FILES[:-1], FILES[1:]))
print("Input:", DATA_DIR)
print("Adjacent pairs:", len(PAIRS))
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/raftup_cutoff_numba")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/raftup_cutoff_mpl")
os.environ.setdefault("OMP_NUM_THREADS", "4")
sys.path.insert(0, str(Path("/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src")))
import torch, traceback
from raftup._cutoff_selection import compute_scanit_cost, run_automated_alignment
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
torch.set_num_threads(4)
SCANIT = dict(n_epoch=3500, n_h=32, seed=0, device="cpu")
LABEL_KEY = "FUSEmap_main_level"
GPR_RADIUS = 100.0
SAVE_FEATURE_COST = False

import hashlib, inspect
from raftup import _cutoff_selection as current_cs
source_root = Path("/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src/raftup")
assert Path(current_cs.__file__).resolve() == (source_root / "_cutoff_selection.py").resolve(), "Restart kernel and import local repo"
assert inspect.signature(current_cs.run_automated_alignment).parameters["technology"].default == "dlpfc"
assert "pixels_per_100um" in inspect.signature(current_cs.run_automated_alignment).parameters, "Restart kernel: outdated main function"
SOURCE_FINGERPRINTS = {str(p.relative_to(source_root)): hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in sorted(source_root.rglob("*")) if p.is_file() and p.suffix in {".py", ".pkl"}}


## Evaluation and plotting helpers

In [ ]:
def input_identity(path, obj):
    # Catch a different dataset, cell order, coordinates, or changed source file.
    xy = np.ascontiguousarray(obj.obsm["spatial"], dtype=np.float64)
    digest = hashlib.sha256(xy.tobytes())
    digest.update(json.dumps(obj.obs_names.tolist()).encode())
    digest.update(json.dumps(obj.var_names.tolist()).encode())
    return dict(path=str(path.resolve()), file_size=path.stat().st_size,
                mtime_ns=path.stat().st_mtime_ns, shape=list(obj.shape),
                coordinate_and_order_sha256=digest.hexdigest())

def as_csr(matrix):
    return matrix.tocsr() if sparse.issparse(matrix) else sparse.csr_matrix(matrix)

def rigid_from_P(P, x, y):
    """Weighted proper rigid fit: y @ R + t approximates x (B -> A)."""
    P = as_csr(P)
    if not P.nnz or not np.isfinite(P.data).all() or np.any(P.data < 0):
        raise ValueError("Empty/invalid alignment cannot define a rigid transform")
    wa = np.asarray(P.sum(1)).ravel(); wb = np.asarray(P.sum(0)).ravel()
    if (wa > 0).sum() < 3 or (wb > 0).sum() < 3:
        raise ValueError("Fewer than three matched points: rigid fit not supported")
    mass = float(wa.sum())
    cx = (wa[:, None]*x).sum(0)/mass; cy = (wb[:, None]*y).sum(0)/mass
    H = (y-cy).T @ (P.T @ (x-cx))
    U, s, Vt = np.linalg.svd(H)
    if s[0] <= 0 or s[-1]/s[0] < 1e-10:
        raise ValueError("Degenerate correspondence geometry: refusing arbitrary rotation")
    fix = np.eye(2); fix[-1, -1] = np.linalg.det(U @ Vt)
    R = U @ fix @ Vt
    t = cx - cy @ R
    return R, t
def gpr_sparse(P, x, y, radius, p_cut=1e-19):
    """Same formula as GPR_original, P_cut=p_cut; sparse radius neighbors."""
    valid = np.asarray(P.sum(1)).ravel() >= p_cut
    if not valid.any():
        return float("nan")
    P = P[valid].copy(); P.data[P.data < 1e-100] = 0; P.eliminate_zeros()
    mass = np.asarray(P.sum(1)).ravel()
    W = sparse.diags(1/np.where(mass > 0, mass, 1)) @ P
    def neighbors(z):
        lists = cKDTree(z).query_ball_point(z, radius)
        rr = np.repeat(np.arange(len(z)), [len(v) for v in lists])
        cc = np.concatenate(lists)
        return sparse.csr_matrix((np.ones(len(rr)), (rr, cc)), shape=(len(z), len(z)))
    NA = neighbors(x[valid]); NB = neighbors(y)
    NA = sparse.diags(1/np.asarray(NA.sum(1)).ravel()) @ NA
    total = 0.
    for start in range(0, len(mass), 128):
        stop = min(start+128, len(mass))
        total += float(((NA[start:stop] @ W) @ NB).multiply(W[start:stop]).sum())
    return total/len(mass)

def evaluate(P, a, b):
    wa = np.asarray(P.sum(1)).ravel(); wb = np.asarray(P.sum(0)).ravel()
    rows = np.flatnonzero(wa > 0)
    if len(rows):
        cols = np.asarray(P[rows].argmax(1)).ravel()
        la = a.obs[LABEL_KEY].astype(str).to_numpy()
        lb = b.obs[LABEL_KEY].astype(str).to_numpy()
        laa = float(np.mean(la[rows] == lb[cols]))
    else:
        laa = float("nan")
    return dict(LAA=laa, GPR=gpr_sparse(P, a.obsm["spatial"], b.obsm["spatial"], GPR_RADIUS),
                coverage_A=float((wa > 0).mean()), coverage_B=float((wb > 0).mean()))

def plot_pair(P, a, b, folder, pair_name):
    x = np.asarray(a.obsm["spatial"], float); y = np.asarray(b.obsm["spatial"], float)
    fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    y_fit = None; reason = None
    try:
        R, t = rigid_from_P(P, x, y); y_fit = y @ R + t
    except ValueError as e:
        reason = str(e)
    for ax, yy, title in ((axs[0], y, "Input: independently rotated"),
                          (axs[1], y if y_fit is None else y_fit, "Rigid fit B to A")):
        ax.scatter(x[:,0], x[:,1], s=1.5, alpha=.4, label="A")
        ax.scatter(yy[:,0], yy[:,1], s=1.5, alpha=.4, label="B")
        ax.set_aspect("equal"); ax.set_xlabel("X (source units)"); ax.set_ylabel("Y (source units)")
        ax.set_title(title); ax.legend(markerscale=4)
    if y_fit is not None:
        rows = np.flatnonzero(np.asarray(P.sum(1)).ravel() > 0)
        rng = np.random.default_rng(0)
        rows = np.sort(rng.choice(rows, min(300, len(rows)), replace=False))
        cols = np.asarray(P[rows].argmax(1)).ravel()
        segments = np.stack([x[rows], y_fit[cols]], axis=1)
        axs[1].add_collection(LineCollection(segments, colors="black", linewidths=.35, alpha=.25))
    else:
        axs[1].set_title("No supported rigid fit: " + reason, fontsize=9)
    fig.suptitle(pair_name + " — all cells; at most 300 argmax links displayed")
    fig.savefig(folder / "alignment.png", dpi=180); plt.close(fig)


## Run all adjacent pairs (expensive)
Each pair is independently resumable. Keep DATA_DIR and RUN_DIR paired; change RUN_DIR if settings change.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
summary = []
for path_a, path_b in PAIRS:
    pair_name = path_a.stem + "__" + path_b.stem
    folder = RUN_DIR / pair_name; folder.mkdir(exist_ok=True)
    print("Starting", pair_name, flush=True)
    a = b = aa = bb = cost = alignment = P = None
    try:
        a = ad.read_h5ad(path_a); b = ad.read_h5ad(path_b)
        assert LABEL_KEY in a.obs and LABEL_KEY in b.obs
        config = dict(A=input_identity(path_a, a), B=input_identity(path_b, b),
                      technology="merfish", source_fingerprints=SOURCE_FINGERPRINTS, scanit=SCANIT, label_key=LABEL_KEY,
                      gpr_radius=GPR_RADIUS, gpr_P_cut=1e-19)
        config_path = folder / "run_config.json"
        if config_path.exists():
            if json.loads(config_path.read_text()) != config:
                raise ValueError("Inputs/settings differ from cache. Choose a new RUN_DIR.")
        elif (folder / "alignment_matrix.npy").exists():
            raise ValueError("Matrix without provenance: choose a new RUN_DIR")
        else:
            config_path.write_text(json.dumps(config, indent=2))
        timing_path = folder / "timings.json"
        timings = json.loads(timing_path.read_text()) if timing_path.exists() else {}
        matrix_path = folder / "alignment_matrix.npy"
        if not matrix_path.exists():
            aa, bb = a.copy(), b.copy()
            for obj in (aa, bb):
                obj.obs = obj.obs.iloc[:, :0].copy(); obj.uns.clear()
            cost_path = folder / "feature_cost.npy"
            if cost_path.exists():
                cost = np.load(cost_path, mmap_mode="r")
            else:
                started = time.perf_counter()
                with (folder / "scanit.log").open("w", buffering=1) as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                    cost = compute_scanit_cost(aa, bb, technology="merfish", **SCANIT)
                timings["scanit_seconds"] = time.perf_counter()-started
                timing_path.write_text(json.dumps(timings, indent=2))
                if SAVE_FEATURE_COST:
                    np.save(cost_path, cost)
            started = time.perf_counter()
            with (folder / "alignment.log").open("w", buffering=1) as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                alignment, decision = run_automated_alignment(aa, bb, cost, technology="merfish", pixels_per_100um=100)
            timings["alignment_seconds"] = time.perf_counter()-started
            # Save atomically: an interrupted write cannot become a valid cache.
            with (folder / "alignment_matrix.tmp").open("wb") as f:
                np.save(f, alignment)
            (folder / "cutoff_decision.json").write_text(json.dumps(decision, indent=2))
            timing_path.write_text(json.dumps(timings, indent=2))
            (folder / "alignment_matrix.tmp").replace(matrix_path)
            del cost, aa, bb, alignment
            cost = aa = bb = alignment = None
        alignment = np.load(matrix_path, mmap_mode="r")
        if alignment.shape != (len(a), len(b)):
            raise ValueError("Wrong matrix shape")
        P = as_csr(alignment)
        if not np.isfinite(P.data).all() or np.any(P.data < 0):
            raise ValueError("Invalid matrix weights")
        # Sparse copy saves memory when assembling the reconstruction later.
        sparse.save_npz(folder / "alignment_matrix_sparse.npz", P)
        (folder / "cell_order.json").write_text(json.dumps(dict(A=a.obs_names.tolist(), B=b.obs_names.tolist())))
        metrics = evaluate(P, a, b)
        status = "completed" if P.nnz else "empty_alignment"
        row = dict(pair=pair_name, n_A=len(a), n_B=len(b), nnz=P.nnz, status=status,
                   **metrics, **timings)
        pd.DataFrame([row]).to_csv(folder / "metrics.csv", index=False)
        (folder / "metrics.json").write_text(json.dumps({k:None if not np.isfinite(v) else v for k,v in metrics.items()}, indent=2))
        plot_pair(P, a, b, folder, pair_name)
        print(row, flush=True)
    except Exception as e:
        row = dict(pair=pair_name, status="failed", error=str(e))
        (folder / "error.txt").write_text(traceback.format_exc())
        print("FAILED:", pair_name, str(e), flush=True)
    finally:
        summary.append(row)
        pd.DataFrame(summary).to_csv(RUN_DIR / "all_pair_metrics.csv", index=False)
        a = b = aa = bb = cost = alignment = P = None
        gc.collect()
pd.DataFrame(summary)


## Summary

In [ ]:
results = pd.read_csv(RUN_DIR / "all_pair_metrics.csv")
display(results)
print("PNG and matrix folders:", RUN_DIR)
print("All pairs completed:", len(results) == len(PAIRS) and results.status.eq("completed").all())